# 🏭 Production Planning — ML Pipeline Notebook

**Project:** `prod_plan_mlmodel`  
**Goal:** Predict order-quantity forecasts, material shortage risk, and overload probability  
**Models trained:**
- `model_order_qty_forecast` — Gradient Boosted Regressor  
- `model_shortage_classifier` — Random Forest Classifier  
- `model_shortage_probability` — Logistic Regression  

---


## 0. Imports & Configuration

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, roc_auc_score, confusion_matrix,
    ConfusionMatrixDisplay
)
import pickle, json, os

# Plotting style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.3f}".format)

DATA_DIR   = "data/"          # adjust if running from repo root
OUTPUT_DIR = "ml_artifacts/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ All libraries loaded successfully")


---
## 1. Data Loading

We have **10 datasets** covering the full manufacturing value chain:

| Dataset | Rows | Description |
|---|---|---|
| `customer_order_master` | 237 | Customer orders, delivery status, VIP flags |
| `demand_forecast_sales` | 180 | Weekly demand forecasts with shortage probability |
| `production_orders` | 237 | Planned vs actual production, delays |
| `material_replenishment` | 180 | Stock levels, reorder points, shortage flags |
| `scrap_quality_inspection` | 237 | Defect rates, scrap cost, rework |
| `work_centre_utilization` | 1575 | Shift-level WC utilization and downtime |
| `machine_master` | 11 | Machine specs and maintenance schedule |
| `material_master` | 12 | Material groups, costs, safety stock |
| `supplier_master` | 10 | Supplier OTIF, risk scores |


In [ ]:

# ── Load all datasets ──────────────────────────────────────────────────────
customer_orders  = pd.read_csv(DATA_DIR + "customer_order_master.csv")
demand_forecast  = pd.read_csv(DATA_DIR + "demand_forecast_sales_fixed.csv")
production_orders= pd.read_csv(DATA_DIR + "production_orders_fixed.csv")
material_replen  = pd.read_csv(DATA_DIR + "material_replenishment.csv")
scrap_quality    = pd.read_csv(DATA_DIR + "scrap_quality_inspection.csv")
wc_utilization   = pd.read_csv(DATA_DIR + "work_centre_utilization_fixed.csv")
machine_master   = pd.read_csv(DATA_DIR + "machine_master.csv")
material_master  = pd.read_csv(DATA_DIR + "material_master.csv")
supplier_master  = pd.read_csv(DATA_DIR + "supplier_master.csv")

datasets = {
    "customer_orders" : customer_orders,
    "demand_forecast" : demand_forecast,
    "production_orders": production_orders,
    "material_replen" : material_replen,
    "scrap_quality"   : scrap_quality,
    "wc_utilization"  : wc_utilization,
    "machine_master"  : machine_master,
    "material_master" : material_master,
    "supplier_master" : supplier_master,
}

print("Dataset shapes:")
for name, df in datasets.items():
    print(f"  {name:<25} {str(df.shape):<15} columns: {list(df.columns)[:4]} ...")


---
## 2. Exploratory Data Analysis (EDA)

### 2.1 Missing Values & Data Quality


In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, (name, df) in enumerate(datasets.items()):
    null_pct = df.isnull().mean() * 100
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        axes[i].text(0.5, 0.5, "No missing values ✅", ha="center", va="center",
                     fontsize=13, transform=axes[i].transAxes)
        axes[i].set_title(name, fontweight="bold")
        axes[i].axis("off")
    else:
        null_pct.sort_values().plot(kind="barh", ax=axes[i], color="#e07b54")
        axes[i].set_title(f"{name} — Missing %", fontweight="bold")
        axes[i].set_xlabel("% Missing")

plt.tight_layout()
plt.suptitle("Missing Value Analysis by Dataset", fontsize=15, fontweight="bold", y=1.01)
plt.show()


### 2.2 Demand Forecast — Weekly Forecast vs Historical Average

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Weekly forecast qty by scenario
df_fc = demand_forecast.groupby(["week_no", "scenario"])["forecast_qty"].mean().reset_index()
for scenario, grp in df_fc.groupby("scenario"):
    axes[0].plot(grp["week_no"], grp["forecast_qty"], marker="o", label=scenario, linewidth=2)
axes[0].set_title("Avg Forecast Qty per Week by Scenario", fontweight="bold")
axes[0].set_xlabel("Week No"); axes[0].set_ylabel("Forecast Qty")
axes[0].legend()

# Forecast vs historical
axes[1].scatter(demand_forecast["historical_avg_qty"], demand_forecast["forecast_qty"],
                alpha=0.6, c=demand_forecast["confidence_score"], cmap="RdYlGn",
                edgecolors="grey", linewidths=0.3, s=50)
lim = max(demand_forecast[["forecast_qty","historical_avg_qty"]].max())
axes[1].plot([0, lim], [0, lim], "k--", linewidth=1, label="Perfect forecast")
axes[1].set_xlabel("Historical Avg Qty"); axes[1].set_ylabel("Forecast Qty")
axes[1].set_title("Forecast vs Historical (color = confidence)", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"Shortage events in forecast: {demand_forecast['shortage_probability'].gt(0.5).sum()} / {len(demand_forecast)}")


### 2.3 Production Orders — Delays & Status

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Order status distribution
status_cnt = production_orders["order_status"].value_counts()
axes[0].pie(status_cnt, labels=status_cnt.index, autopct="%1.1f%%",
            colors=sns.color_palette("muted", len(status_cnt)), startangle=90)
axes[0].set_title("Production Order Status", fontweight="bold")

# Start delay distribution
production_orders["start_delay_hrs"].plot(kind="hist", bins=25, ax=axes[1],
                                          color="#5b8db8", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.5, label="On-time")
axes[1].set_title("Start Delay Distribution (hrs)", fontweight="bold")
axes[1].set_xlabel("Hours"); axes[1].legend()

# Overload probability by work centre
wc_overload = production_orders.groupby("work_centre")["overload_probability"].mean().sort_values(ascending=False)
wc_overload.head(10).plot(kind="bar", ax=axes[2], color="#e07b54", edgecolor="white")
axes[2].set_title("Avg Overload Probability by Work Centre", fontweight="bold")
axes[2].set_xlabel("Work Centre"); axes[2].set_ylabel("Avg Overload Prob")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### 2.4 Work Centre Utilization — Heatmap by Shift

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Utilization by WC and shift
pivot = wc_utilization.pivot_table(values="utilization_pct",
                                   index="work_centre", columns="shift", aggfunc="mean")
sns.heatmap(pivot, ax=axes[0], cmap="YlOrRd", annot=True, fmt=".0f",
            linewidths=0.4, cbar_kws={"label": "Utilization %"})
axes[0].set_title("Avg Utilisation % by Work Centre & Shift", fontweight="bold")

# Downtime by reason
dt_reason = wc_utilization.groupby("downtime_reason")["downtime_hrs"].sum().sort_values(ascending=False).head(8)
dt_reason.plot(kind="barh", ax=axes[1], color="#5b8db8", edgecolor="white")
axes[1].set_title("Total Downtime Hours by Reason (Top 8)", fontweight="bold")
axes[1].set_xlabel("Total Downtime Hrs")

plt.tight_layout()
plt.show()


### 2.5 Scrap & Quality Inspection

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Defect type breakdown
defect_cnt = scrap_quality["defect_type"].value_counts()
defect_cnt.plot(kind="bar", ax=axes[0], color=sns.color_palette("tab10"), edgecolor="white")
axes[0].set_title("Defect Type Frequency", fontweight="bold")
axes[0].set_xlabel("Defect Type"); axes[0].tick_params(axis="x", rotation=45)

# Scrap cost distribution
scrap_quality["scrap_cost_eur"].plot(kind="hist", bins=30, ax=axes[1],
                                     color="#e07b54", edgecolor="white")
axes[1].set_title("Scrap Cost Distribution (EUR)", fontweight="bold")
axes[1].set_xlabel("Scrap Cost (EUR)")

# Scrap rate by work centre
wc_scrap = scrap_quality.groupby("work_centre")["scrap_rate_pct"].mean().sort_values(ascending=False)
wc_scrap.plot(kind="bar", ax=axes[2], color="#5b8db8", edgecolor="white")
axes[2].set_title("Avg Scrap Rate % by Work Centre", fontweight="bold")
axes[2].set_xlabel("Work Centre"); axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Total scrap cost: EUR {scrap_quality['scrap_cost_eur'].sum():,.0f}")
print(f"Avg defect rate: {scrap_quality['defect_rate_pct'].mean():.2f}%")


### 2.6 Supplier & Material Risk Overview

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Supplier OTIF vs Risk Score
sc = axes[0].scatter(supplier_master["otif_pct"], supplier_master["risk_score_m5"],
                     s=supplier_master["avg_lead_time_days"]*15,
                     c=supplier_master["scrap_contribution_pct"], cmap="RdYlGn_r",
                     alpha=0.8, edgecolors="grey", linewidths=0.5)
for _, row in supplier_master.iterrows():
    axes[0].annotate(row["supplier_id"], (row["otif_pct"], row["risk_score_m5"]),
                     fontsize=8, ha="left", va="bottom")
axes[0].set_xlabel("OTIF %"); axes[0].set_ylabel("Risk Score")
axes[0].set_title("Supplier: OTIF vs Risk Score
(size=lead time, color=scrap contribution)", fontweight="bold")
plt.colorbar(sc, ax=axes[0], label="Scrap Contribution %")

# Material replenishment — shortage flags
shortage_by_mat = material_replen.groupby("material_no")["shortage_flag"].sum().sort_values(ascending=False)
shortage_by_mat.plot(kind="bar", ax=axes[1], color="#e07b54", edgecolor="white")
axes[1].set_title("Shortage Flag Count by Material", fontweight="bold")
axes[1].set_xlabel("Material No"); axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


---
## 3. Feature Engineering

We build a **master modelling table** by joining all datasets on `material_no`, `week_no`, and `scenario`.

**Engineered features:**
- `qty_deviation` — forecast vs historical gap
- `stock_coverage_weeks` — stock / weekly forecast
- `delay_ratio` — actual vs planned hours ratio
- `quality_risk_score` — combined scrap + rework signal
- `supplier_risk_flag` — OTIF < 85 or risk score > 60


In [ ]:

# ── Base: demand forecast ─────────────────────────────────────────────────
df = demand_forecast.copy()

# ── Merge material replenishment ──────────────────────────────────────────
df = df.merge(
    material_replen[["week_no","material_no","scenario","stock_level","shortage_flag",
                      "shortage_probability","replenishment_order_qty","supplier_otif_pct"]],
    on=["week_no","material_no","scenario"], how="left",
    suffixes=("","_replen")
)

# ── Merge production orders (aggregate to material+week level) ─────────────
prod_agg = production_orders.groupby(["week_no","material_no","scenario"]).agg(
    avg_start_delay=("start_delay_hrs","mean"),
    avg_finish_delay=("finish_delay_hrs","mean"),
    avg_overload_prob=("overload_probability","mean"),
    avg_throughput_dev=("throughput_deviation_pct","mean"),
    n_orders=("production_order_no","count")
).reset_index()
df = df.merge(prod_agg, on=["week_no","material_no","scenario"], how="left")

# ── Merge scrap quality (aggregate to material+week level) ─────────────────
scrap_agg = scrap_quality.groupby(["week_no","material_no","scenario"]).agg(
    avg_defect_rate=("defect_rate_pct","mean"),
    avg_scrap_rate=("scrap_rate_pct","mean"),
    avg_scrap_risk=("scrap_risk_probability","mean"),
    total_scrap_cost=("scrap_cost_eur","sum")
).reset_index()
df = df.merge(scrap_agg, on=["week_no","material_no","scenario"], how="left")

# ── Merge material master ──────────────────────────────────────────────────
df = df.merge(
    material_master[["material_no","unit_cost_eur","reorder_point","safety_stock","standard_batch_qty"]],
    on="material_no", how="left"
)

# ── Merge supplier master ──────────────────────────────────────────────────
df = df.merge(
    supplier_master[["supplier_id","otif_pct","avg_lead_time_days","risk_score_m5","scrap_contribution_pct"]],
    on="supplier_id", how="left"
)

# ── Feature engineering ────────────────────────────────────────────────────
df["qty_deviation"]        = df["forecast_qty"] - df["historical_avg_qty"]
df["qty_deviation_pct"]    = df["qty_deviation"] / (df["historical_avg_qty"] + 1e-6)
df["stock_coverage_weeks"] = df["stock_level"] / (df["forecast_qty"] + 1e-6)
df["below_safety_stock"]   = (df["stock_level"] < df["safety_stock"]).astype(int)
df["below_reorder"]        = (df["stock_level"] < df["reorder_point"]).astype(int)
df["quality_risk_score"]   = df["avg_scrap_rate"].fillna(0) + df["avg_defect_rate"].fillna(0)
df["supplier_risk_flag"]   = ((df["otif_pct"] < 85) | (df["risk_score_m5"] > 60)).astype(int)
df["delay_signal"]         = df["avg_start_delay"].fillna(0) + df["avg_finish_delay"].fillna(0)

# Label encode categoricals
le_material = LabelEncoder()
le_scenario = LabelEncoder()
df["material_encoded"] = le_material.fit_transform(df["material_no"])
df["scenario_encoded"] = le_scenario.fit_transform(df["scenario"])

print(f"Master modelling table: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(3)


In [ ]:

# Save label encoders (mirrors ml_artifacts/)
with open(OUTPUT_DIR + "label_encoder_material.pkl","wb") as f: pickle.dump(le_material, f)
with open(OUTPUT_DIR + "label_encoder_scenario.pkl","wb") as f: pickle.dump(le_scenario, f)

# Correlation heatmap of numeric features
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, annot=False,
            linewidths=0.3, cbar_kws={"shrink":0.8})
plt.title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


---
## 4. Model 1 — Order Quantity Forecast (Regression)

**Target:** `forecast_qty`  
**Algorithm:** Gradient Boosted Regressor  
**Purpose:** Predict weekly demand per material


In [ ]:

FEATURE_COLS_REG = [
    "week_no", "material_encoded", "scenario_encoded",
    "historical_avg_qty", "seasonal_index", "deviation_flag",
    "confidence_score", "supplier_lead_time_days",
    "stock_level", "reorder_point", "replenishment_order_qty",
    "avg_start_delay", "avg_finish_delay", "avg_overload_prob",
    "avg_throughput_dev", "n_orders",
    "avg_defect_rate", "avg_scrap_rate", "avg_scrap_risk",
    "unit_cost_eur", "safety_stock", "standard_batch_qty",
    "otif_pct", "avg_lead_time_days", "risk_score_m5",
    "qty_deviation", "qty_deviation_pct", "stock_coverage_weeks",
    "below_safety_stock", "below_reorder",
    "quality_risk_score", "supplier_risk_flag", "delay_signal"
]

TARGET_REG = "forecast_qty"

df_model = df[FEATURE_COLS_REG + [TARGET_REG]].dropna()
X = df_model[FEATURE_COLS_REG]
y = df_model[TARGET_REG]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gbr = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.08, max_depth=4,
    subsample=0.85, min_samples_leaf=3, random_state=42
)
gbr.fit(X_train, y_train)
y_pred = gbr.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2   = r2_score(y_test, y_pred)

print(f"📊 Order Qty Forecast — Test Metrics")
print(f"   MAE  : {mae:.2f}")
print(f"   RMSE : {rmse:.2f}")
print(f"   R²   : {r2:.4f}")

# Cross-val
cv_r2 = cross_val_score(gbr, X, y, cv=5, scoring="r2")
print(f"   CV R² (5-fold): {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.6, color="#5b8db8", edgecolors="none", s=40)
lim = max(y_test.max(), y_pred.max())
axes[0].plot([0,lim],[0,lim],"r--", linewidth=1.5, label="Perfect")
axes[0].set_xlabel("Actual"); axes[0].set_ylabel("Predicted")
axes[0].set_title("Actual vs Predicted — Forecast Qty", fontweight="bold")
axes[0].legend()

# Residuals
residuals = y_test.values - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color="#e07b54", edgecolors="none", s=35)
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")
axes[1].set_title("Residual Plot", fontweight="bold")

# Feature importance
feat_imp = pd.Series(gbr.feature_importances_, index=FEATURE_COLS_REG).sort_values(ascending=True).tail(15)
feat_imp.plot(kind="barh", ax=axes[2], color="#5b8db8", edgecolor="white")
axes[2].set_title("Top 15 Feature Importances", fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:

# Save model
with open(OUTPUT_DIR + "model_order_qty_forecast.pkl","wb") as f: pickle.dump(gbr, f)
print("✅ model_order_qty_forecast.pkl saved")


---
## 5. Model 2 — Shortage Classifier (Binary Classification)

**Target:** `shortage_flag` (0 = no shortage, 1 = shortage)  
**Algorithm:** Random Forest Classifier  
**Purpose:** Predict whether a material will face shortage in a given week


In [ ]:

FEATURE_COLS_CLS = [
    "week_no", "material_encoded", "scenario_encoded",
    "forecast_qty", "historical_avg_qty", "seasonal_index",
    "confidence_score", "supplier_lead_time_days",
    "stock_level", "reorder_point", "replenishment_order_qty",
    "supplier_otif_pct", "avg_start_delay", "avg_overload_prob",
    "avg_throughput_dev", "avg_scrap_risk", "quality_risk_score",
    "unit_cost_eur", "safety_stock",
    "otif_pct", "risk_score_m5",
    "qty_deviation_pct", "stock_coverage_weeks",
    "below_safety_stock", "below_reorder",
    "supplier_risk_flag", "delay_signal"
]

TARGET_CLS = "shortage_flag"

df_cls = df[FEATURE_COLS_CLS + [TARGET_CLS]].dropna()
X_c = df_cls[FEATURE_COLS_CLS]
y_c = df_cls[TARGET_CLS]

print(f"Class distribution:\n{y_c.value_counts()}")

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2,
                                                     stratify=y_c, random_state=42)

rfc = RandomForestClassifier(
    n_estimators=200, max_depth=8, min_samples_leaf=2,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rfc.fit(X_tr_c, y_tr_c)
y_pred_c = rfc.predict(X_te_c)
y_prob_c = rfc.predict_proba(X_te_c)[:,1]

print("\n📊 Shortage Classifier — Test Metrics")
print(classification_report(y_te_c, y_pred_c, target_names=["No Shortage","Shortage"]))
try:
    print(f"   ROC-AUC: {roc_auc_score(y_te_c, y_prob_c):.4f}")
except Exception as e:
    print(f"   ROC-AUC: {e}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y_te_c, y_pred_c)
ConfusionMatrixDisplay(cm, display_labels=["No Shortage","Shortage"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix — Shortage Classifier", fontweight="bold")

# Feature importance
fi_c = pd.Series(rfc.feature_importances_, index=FEATURE_COLS_CLS).sort_values(ascending=True).tail(15)
fi_c.plot(kind="barh", ax=axes[1], color="#5b8db8", edgecolor="white")
axes[1].set_title("Top 15 Feature Importances", fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:

with open(OUTPUT_DIR + "model_shortage_classifier.pkl","wb") as f: pickle.dump(rfc, f)
print("✅ model_shortage_classifier.pkl saved")


---
## 6. Model 3 — Shortage Probability (Calibrated Logistic Regression)

**Target:** `shortage_probability` (continuous 0–1)  
**Algorithm:** Logistic Regression (calibrated)  
**Purpose:** Output a probability score for shortages to drive replenishment alerts


In [ ]:

FEATURE_COLS_PROB = [
    "week_no", "material_encoded", "scenario_encoded",
    "forecast_qty", "historical_avg_qty", "seasonal_index",
    "confidence_score", "supplier_lead_time_days",
    "stock_level", "reorder_point",
    "supplier_otif_pct", "avg_overload_prob", "avg_scrap_risk",
    "qty_deviation_pct", "stock_coverage_weeks",
    "below_safety_stock", "below_reorder",
    "supplier_risk_flag", "otif_pct", "risk_score_m5"
]

TARGET_PROB = "shortage_probability"

df_prob = df[FEATURE_COLS_PROB + [TARGET_PROB]].dropna()
X_p = df_prob[FEATURE_COLS_PROB]
y_p = df_prob[TARGET_PROB]

X_tr_p, X_te_p, y_tr_p, y_te_p = train_test_split(X_p, y_p, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_ps = scaler.fit_transform(X_tr_p)
X_te_ps = scaler.transform(X_te_p)

# Binarise target for logistic (threshold 0.5)
y_tr_pb = (y_tr_p >= 0.5).astype(int)
y_te_pb = (y_te_p >= 0.5).astype(int)

lr = LogisticRegression(max_iter=500, class_weight="balanced", C=0.5, random_state=42)
lr.fit(X_tr_ps, y_tr_pb)
y_prob_p = lr.predict_proba(X_te_ps)[:,1]

print("📊 Shortage Probability Model — Test Metrics")
print(f"   MAE  (prob vs actual): {mean_absolute_error(y_te_p, y_prob_p):.4f}")
print(f"   ROC-AUC             : {roc_auc_score(y_te_pb, y_prob_p):.4f}")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted probability distribution
axes[0].hist(y_prob_p[y_te_pb==0], bins=25, alpha=0.7, label="No Shortage", color="#5b8db8", edgecolor="white")
axes[0].hist(y_prob_p[y_te_pb==1], bins=25, alpha=0.7, label="Shortage", color="#e07b54", edgecolor="white")
axes[0].set_xlabel("Predicted Probability"); axes[0].set_ylabel("Count")
axes[0].set_title("Predicted Shortage Probability Distribution", fontweight="bold")
axes[0].legend()

# Coefficient plot
coef = pd.Series(lr.coef_[0], index=FEATURE_COLS_PROB).sort_values()
colors = ["#e07b54" if c > 0 else "#5b8db8" for c in coef]
coef.plot(kind="barh", ax=axes[1], color=colors, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Logistic Regression Coefficients", fontweight="bold")

plt.tight_layout()
plt.show()


In [ ]:

with open(OUTPUT_DIR + "model_shortage_probability.pkl","wb") as f: pickle.dump((lr, scaler), f)
print("✅ model_shortage_probability.pkl saved")


---
## 7. Save Feature Columns & Training Metrics

Save `feature_columns.json` and `training_metrics.json` to `ml_artifacts/` (mirrors the existing project structure).


In [ ]:

feature_columns = {
    "regression"  : FEATURE_COLS_REG,
    "classifier"  : FEATURE_COLS_CLS,
    "probability" : FEATURE_COLS_PROB
}
with open(OUTPUT_DIR + "feature_columns.json","w") as f:
    json.dump(feature_columns, f, indent=2)

y_pred_reg   = gbr.predict(X_test)
y_pred_cls   = rfc.predict(X_te_c)
y_prob_cls   = rfc.predict_proba(X_te_c)[:,1]

training_metrics = {
    "model_order_qty_forecast": {
        "mae" : round(mean_absolute_error(y_test, y_pred_reg), 4),
        "rmse": round(mean_squared_error(y_test, y_pred_reg, squared=False), 4),
        "r2"  : round(r2_score(y_test, y_pred_reg), 4)
    },
    "model_shortage_classifier": {
        "roc_auc" : round(roc_auc_score(y_te_c, y_prob_cls), 4),
    },
    "model_shortage_probability": {
        "mae"     : round(mean_absolute_error(y_te_p, y_prob_p), 4),
        "roc_auc" : round(roc_auc_score(y_te_pb, y_prob_p), 4)
    }
}
with open(OUTPUT_DIR + "training_metrics.json","w") as f:
    json.dump(training_metrics, f, indent=2)

print("✅ feature_columns.json saved")
print("✅ training_metrics.json saved")
print()
print(json.dumps(training_metrics, indent=2))


---
## 8. Inference — Sample Prediction

Simulate what `api_server.py` does: load the three artifacts and score a single record.


In [ ]:

# Load all artifacts
with open(OUTPUT_DIR + "model_order_qty_forecast.pkl","rb")   as f: model_reg  = pickle.load(f)
with open(OUTPUT_DIR + "model_shortage_classifier.pkl","rb")  as f: model_cls  = pickle.load(f)
with open(OUTPUT_DIR + "model_shortage_probability.pkl","rb") as f: prob_tuple = pickle.load(f)
model_prob, prob_scaler = prob_tuple

with open(OUTPUT_DIR + "label_encoder_material.pkl","rb") as f: le_mat = pickle.load(f)
with open(OUTPUT_DIR + "label_encoder_scenario.pkl","rb")  as f: le_scn = pickle.load(f)
with open(OUTPUT_DIR + "feature_columns.json","r") as f: feat_cols = json.load(f)

# Build a sample row from the last week of data
sample_raw = df.sort_values("week_no", ascending=False).iloc[[0]].copy()

def score_sample(row):
    results = {}

    # Regression
    X_r = row[feat_cols["regression"]].fillna(0)
    results["forecast_qty_pred"] = round(float(model_reg.predict(X_r)[0]), 2)

    # Classifier
    X_c = row[feat_cols["classifier"]].fillna(0)
    results["shortage_flag_pred"] = int(model_cls.predict(X_c)[0])
    results["shortage_flag_proba"] = round(float(model_cls.predict_proba(X_c)[0][1]), 4)

    # Probability model
    X_p = row[feat_cols["probability"]].fillna(0)
    X_ps = prob_scaler.transform(X_p)
    results["shortage_probability_pred"] = round(float(model_prob.predict_proba(X_ps)[0][1]), 4)

    return results

result = score_sample(sample_raw)
print("🔮 Sample Inference Result")
print(f"  Material          : {sample_raw['material_no'].values[0]}")
print(f"  Week              : {sample_raw['week_no'].values[0]}")
print(f"  Scenario          : {sample_raw['scenario'].values[0]}")
print()
for k, v in result.items():
    print(f"  {k:<35}: {v}")


---
## 9. Summary & Next Steps

### What was built
| Artifact | Purpose |
|---|---|
| `model_order_qty_forecast.pkl` | GBR regression — predicts weekly forecast qty |
| `model_shortage_classifier.pkl` | RF classifier — predicts shortage yes/no |
| `model_shortage_probability.pkl` | Logistic regression — shortage probability score |
| `label_encoder_material.pkl` | Encodes material IDs |
| `label_encoder_scenario.pkl` | Encodes scenario labels |
| `feature_columns.json` | Feature lists per model |
| `training_metrics.json` | Evaluation metrics |

### Recommended next steps
1. **Hyperparameter tuning** — Use `GridSearchCV` or `Optuna` for GBR and RF
2. **Time-series cross-validation** — Use `TimeSeriesSplit` instead of random split
3. **SHAP explanations** — Add `shap` for model interpretability
4. **Rolling lag features** — Add lag-1, lag-2 week features for trend capture
5. **API integration** — Models are ready for `api_server.py` inference endpoints
6. **Monitoring** — Track prediction drift with `evidently` or custom metrics
